<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_5_3_Boosting_and_BART.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Boosting and BART: Sequential Learning and Beyond

**Attribution**: Exercises adapted from *Introduction to Modern Statistics (2e)* Chapter 9.6.
Source: [OpenIntro IMS](https://openintro-ims.netlify.app/model-logistic)

**License**: This work is a derivative of [Introduction to Modern Statistics (2e)](https://openintro-ims.netlify.app/) by Mine Çetinkaya-Rundel and Johanna Hardin, used under a [Creative Commons Attribution-ShareAlike 3.0 Unported (CC BY-SA 3.0 US)](https://creativecommons.org/licenses/by-sa/3.0/us/) license.

---

## Introduction
In the previous notebook, we used **Bagging** to build trees independently to reduce variance. In this final notebook, we explore **Boosting**, where trees are built sequentially. Instead of a democracy of independent votes, boosting is like a team of experts where each new tree tries to fix the mistakes of the previous one.

We will conclude with a look at **BART (Bayesian Additive Regression Trees)**, which combines the benefits of both strategies with a robust Bayesian framework.

### Learning Objectives
By the end of this notebook, you will be able to:
1. **Implement** Gradient Boosting and XGBoost models.
2. **Explain** the roles of learning rate ($\lambda$) and shrinkage in boosting.
3. **Contrast** the sequential logic of Boosting with the parallel logic of Bagging.
4. **Understand** the core principles of Bayesian Additive Regression Trees (BART).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Set style
sns.set_context("talk")
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load data
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

---
## 1. Boosting: Learning from Mistakes

Boosting starts with a simple tree (often a "stump") and then builds the next tree to predict the **residuals** (the leftover error) of the ensemble so far. 

### Implementation: XGBoost
XGBoost is one of the most powerful and popular libraries for boosting today.

In [ ]:
# Initialize and fit XGBoost model
xgb = XGBClassifier(
    n_estimators=100, 
    learning_rate=0.05, 
    max_depth=3, 
    random_state=42
)
xgb.fit(X_train, y_train)

print(f"XGBoost Accuracy (Test): {accuracy_score(y_test, xgb.predict(X_test)):.3f}")

### Key Parameters: 
1. **Shrinkage/Learning Rate ($\lambda$ / `learning_rate`)**: Controls how much we trust each new tree. Small values (e.g., 0.01) slow down learning but usually improve accuracy.
2. **Number of Trees ($B$ / `n_estimators`)**: If we have too many trees, boosting will eventually overfit the noise in our data.
3. **Tree Complexity ($d$ / `max_depth`)**: Usually, boosting works best with shallow trees that focus on specific patterns.

---
## 2. Bayesian Additive Regression Trees (BART)

BART is the Bayesian "sibling" of the ensemble methods we've covered. It uses a "sum of trees" approach similar to Boosting, but with a different philosophy.

### Conceptual Overview:
- **Priors**: BART uses Bayesian priors to restrict tree growth. It "regularizes" the trees from the start, preventing any single tree from dominating the model.
- **Perturbation**: Instead of building new trees from scratch, BART slightly changes ("perturbates") the existing trees in each iteration (e.g., changing a split point or pruning a leaf).
- **Uncertainty**: Unlike Boosting or Random Forests, BART naturally provides **Credible Intervals**—allowing you to know how confident the model is in its malignancy predictions.

**Student Task**: Reflect on why knowing the *confidence* of a tumor prediction might be as important as the prediction itself.

---
## 3. Final Comparison: Which Method Wins?

Let's compare the performance of all three ensemble methods we've learned in this series on the same Breast Cancer test set.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier

# Compare models
models = {
    'Bagging': BaggingClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = accuracy_score(y_test, model.predict(X_test))

pd.Series(results).plot(kind='bar', color=['skyblue', 'salmon', 'lightgreen'])
plt.title("Accuracy Comparison: Bagging vs. RF vs. Boosting")
plt.ylabel("Accuracy Score")
plt.ylim(0.9, 1.0)
plt.show()

## Summary
In this series, we transitioned from the simplicity of a single **Decision Tree** to the massive predictive power of **Ensembles**. 

1. **Decision Trees**: Highly interpretable, but prone to overfitting.
2. **Bagging & Random Forests**: Reduce variance by averaging independent/decorrelated trees.
3. **Boosting & BART**: Minimize bias and refine accuracy by learning from previous mistakes.

In real-world data science, **Random Forests** and **Boosting (XGBoost/LightGBM)** are often the "go-to" models for tabular data like clinical records, financial transactions, and mass masses mass mas...